# From Detections to a Precision-Recall Curve

In Lecture 10 (`DL_025`) you implemented **IoU** and the **bounding box regression loss**.
In this notebook we go one step further and compute the **evaluation metrics** of an object detector:

1. **Match** predictions to ground-truth boxes (IoU ≥ 0.5, every ground-truth box can only be matched once)
2. Sweep the **confidence threshold** and compute **precision** and **recall**
3. Plot the **precision-recall curve** and compute the **Average Precision (AP)**

We work with a single class (*cat*) — for multiple classes you would repeat this per class and average the APs to get the **mAP**.


## 1. The dataset

Five images, seven ground-truth boxes and the ten detections our (fictional) cat detector produced.
Boxes use the `[x, y, w, h]` convention (top-left corner + width/height).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Ground-truth boxes per image, format [x, y, w, h] (top-left corner + width/height)
gt_boxes = {
    "img1": [[50, 60, 120, 100], [300, 200, 150, 120]],
    "img2": [[100, 100, 200, 150]],
    "img3": [[30, 40, 100, 90], [250, 80, 130, 110]],
    "img4": [[120, 150, 180, 140]],
    "img5": [[60, 50, 140, 120]],
}

# Predictions of our (fictional) cat detector: (image_id, box, confidence score)
predictions = [
    ("img1", [55, 65, 115, 95], 0.95),
    ("img1", [310, 210, 140, 115], 0.88),
    ("img1", [200, 100, 80, 60], 0.35),
    ("img2", [110, 110, 190, 145], 0.91),
    ("img2", [105, 95, 210, 160], 0.62),
    ("img3", [35, 45, 95, 85], 0.78),
    ("img3", [260, 90, 120, 100], 0.44),
    ("img4", [125, 155, 170, 135], 0.83),
    ("img4", [0, 0, 60, 50], 0.25),
    ("img5", [150, 200, 100, 80], 0.55),
]


In [ ]:
def show_dataset(gt_boxes, predictions):
    """Draws all images with ground truth (red) and predictions (green + score)."""
    fig, axes = plt.subplots(1, len(gt_boxes), figsize=(4 * len(gt_boxes), 4))
    for ax, img_id in zip(axes, gt_boxes):
        ax.set_title(img_id)
        ax.set_xlim(0, 500); ax.set_ylim(400, 0)   # y-axis flipped like image coordinates
        ax.set_aspect("equal")
        for x, y, w, h in gt_boxes[img_id]:
            ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="red", lw=2))
        for pid, (x, y, w, h), score in [p for p in predictions if p[0] == img_id]:
            ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="green", lw=2, linestyle="--"))
            ax.text(x, y - 5, f"{score:.2f}", color="green", fontsize=10)
    plt.suptitle("red = ground truth, green = predictions of the cat detector")
    plt.show()

show_dataset(gt_boxes, predictions)


## 2. Provided: IoU from Lecture 10

So that everyone starts from a working basis, here is the IoU function from `DL_025`.


In [ ]:
def iou(box_a, box_b):
    """Intersection over Union of two boxes [x, y, w, h] - as implemented in Lecture 10 (DL_025)."""
    ax, ay, aw, ah = box_a
    bx, by, bw, bh = box_b
    ix1, iy1 = max(ax, bx), max(ay, by)
    ix2, iy2 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0

# quick sanity check
assert iou([0, 0, 10, 10], [0, 0, 10, 10]) == 1.0
assert iou([0, 0, 10, 10], [20, 20, 5, 5]) == 0.0
print("iou() ready")


## 3. Task 1 — Match predictions to ground truth

Process the predictions **in descending order of confidence**. A prediction is a **true positive (TP)** if it has
IoU ≥ `iou_threshold` with a *not yet matched* ground-truth box of the same image — otherwise it is a **false positive (FP)**.

Note: a second, weaker detection of an already matched ground-truth box counts as FP — this is exactly what punishes duplicate detections.


In [ ]:
def match_predictions(predictions, gt_boxes, iou_threshold=0.5):
    preds_sorted = sorted(predictions, key=lambda p: p[2], reverse=True)
    matched_gt = set()   # remember which GT boxes are already taken, e.g. as (img_id, index)
    matches = []         # list of (score, is_tp)

    for img_id, box, score in preds_sorted:
        # YOUR CODE HERE:
        # 1. find the not-yet-matched GT box in the same image with the highest IoU
        # 2. if that IoU >= iou_threshold: it is a TP, mark the GT box as matched
        # 3. otherwise it is a FP
        pass

    total_gt = sum(len(boxes) for boxes in gt_boxes.values())
    return matches, total_gt

matches, total_gt = match_predictions(predictions, gt_boxes)
print("matches (score, is_tp):", matches)
print("total ground truth boxes:", total_gt)


In [ ]:
# self-check for task 1
assert total_gt == 7
assert len(matches) == 10
assert sum(1 for _, is_tp in matches if is_tp) == 6, "expected exactly 6 true positives"
assert all(matches[i][0] >= matches[i+1][0] for i in range(len(matches)-1)), "matches must be sorted by score"
print("Task 1 looks good!")


## 4. Task 2 — Precision & recall while sweeping the threshold

Instead of picking one confidence threshold, we take the sorted list of matches and compute precision and recall **after every prediction** —
this is equivalent to lowering the confidence threshold step by step.


In [ ]:
def precision_recall_curve(matches, total_gt):
    # YOUR CODE HERE:
    # Walk through the (already sorted) matches and after every prediction compute
    #   precision = TP_so_far / predictions_so_far
    #   recall    = TP_so_far / total_gt
    # Hint: np.cumsum makes this a 3-liner.
    precision = ...
    recall = ...
    return precision, recall

precision, recall = precision_recall_curve(matches, total_gt)
print("precision:", np.round(precision, 3))
print("recall:   ", np.round(recall, 3))


## 5. Task 3 — PR curve and Average Precision

The AP summarizes the whole curve into a single number: the **area under the precision envelope**
(every precision value is replaced by the maximum precision at equal-or-higher recall, which removes the zigzag).

*Food for thought: why does the curve dip down and then rise again? Look at the prediction with score 0.44.*


In [ ]:
def average_precision(precision, recall):
    # YOUR CODE HERE:
    # 1. build the "precision envelope": replace every precision value by the maximum
    #    of all precision values at the same or higher recall (walk backwards through the array)
    # 2. AP = area under the envelope = sum over (recall[i] - recall[i-1]) * envelope[i]
    ap = ...
    return ap

ap = average_precision(precision, recall)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, "o-", label="PR curve")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.xlim(0, 1.05); plt.ylim(0, 1.05)
plt.title(f"Precision-Recall curve, AP = {ap:.4f}")
plt.grid(True); plt.legend()
plt.show()
print("Average Precision:", round(ap, 4))


In [ ]:
# self-check for task 3
assert abs(ap - 0.8214) < 1e-3, f"expected AP around 0.8214, got {ap}"
print("Task 3 looks good - well done!")


## 6. Wrap-up

* You just implemented the core of what tools like `pycocotools` or Ultralytics report as **mAP50**.
* For **mAP@[0.5:0.95]** (COCO convention) the same computation is repeated for IoU thresholds 0.50, 0.55, …, 0.95 and averaged.
* For multiple classes: compute the AP per class and average → **mAP**.
